# ExtraTrees Regression: Public Research Demonstration

Per-target extremely randomized trees preserve missing values using the estimator's native support. There is no imputation step for this model. A small independent Optuna search illustrates parameter selection; its settings are not the competition settings.

**Scope:** this runnable notebook demonstrates the workflow, not the reported competition score. No competition data or private factors are loaded. See [research limitations](../docs/RESEARCH.md).


## 1. Reproducible synthetic panel

The public generator is unrelated to the withheld feature construction. All three model notebooks use the same ordered row IDs and day groups.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "src" / "quant_portfolio").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
ARTIFACT_DIR = Path(os.environ.get("PORTFOLIO_ARTIFACT_DIR", str(ROOT / "artifacts")))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
from quant_portfolio.data import make_synthetic_panel
from quant_portfolio.blend import weighted_directional_accuracy
from quant_portfolio.models import model_oof, refit_predict
from quant_portfolio.artifacts import save_model_artifacts

panel = make_synthetic_panel(seed=42)
print("Independent synthetic data only:", panel.train.shape, panel.test.shape)


In [ ]:
display(panel.train.head())
print("Training days:", panel.train.day_id.nunique())
print("Missing synthetic covariates:", int(panel.train.filter(like="feature_").isna().sum().sum()))


## 2. Hyperparameter selection

Selection uses five day-grouped folds. The resulting best OOF score is a selection diagnostic, not an independent estimate. The first run is intentionally small for review on a laptop.

An unfinished Optuna batch resumes toward its saved target. After a finished batch, rerunning this cell adds two demo trials. The seed stays fixed at 42. Trial history is persisted, but the sampler RNG stream is not checkpointed. Use only one active process per study; unresolved RUNNING trials trigger an error. Invalid trial controls are rejected before a new database is created.


In [ ]:
from quant_portfolio.search import run_optuna_search
selected = run_optuna_search(
    "extra_trees", panel, ARTIFACT_DIR / "extra_trees_demo.sqlite3",
    initial_trials=3, trials_per_reopen=2, seed=42,
)
params = selected.params
display(selected.table)
print("Chosen demonstration parameters:", params)


## 3. Rebuild aligned OOF predictions

Each evaluation day is absent from that fit's training rows. The feature provider receives training labels only. The same parameters are then used for the full-data refit. Both model calls attach a generation context to their predictions, including the data, implementation, provider, parameters, seed, runtime, and OOF fold protocol.


In [ ]:
MODEL = "extra_trees"
oof = model_oof(MODEL, panel, params, n_splits=5, seed=42)
score = weighted_directional_accuracy(panel.y, oof)
print(f"Synthetic OOF selection score: {score:.6f}")
assert abs(score - selected.score) <= 1e-12
assert oof.index.equals(panel.train.index)


## 4. Full-data fit and prediction artifacts

Fit on all training rows and predict the disjoint unlabeled synthetic test panel. `save_model_artifacts` accepts only a matching OOF/full-refit pair with the context attached by the model functions. It writes an immutable generation under `extra_trees/runs/<run_id>/`, verifies the payloads, and atomically switches `extra_trees/manifest.json` only after the generation is complete. Earlier runs remain available. These are not competition submission files.


In [ ]:
test_raw = refit_predict(MODEL, panel, params, seed=42)
manifest = save_model_artifacts(ARTIFACT_DIR, MODEL, panel, oof, test_raw, params)
display(pd.DataFrame(manifest["files"]).T)
display(test_raw.head())


## Interpretation

There is no held-out synthetic test score: test outcomes are not exposed by the public data API. Model-search results reuse OOF labels, so further independent evaluation would be needed for a generalization claim. Continue to the ensemble notebook only after all three model artifacts have been generated in the same output directory. Legacy schema-1 artifacts are intentionally not upgraded in place; select a fresh output directory and rerun notebooks 01–03. The Optuna study in an older directory is not deleted or modified.
